In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to Customers (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Customers')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='Customers']")))

    # Click the real Add button (aria-label="Add customer", verified in CustomerDirectory.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='Add customer']"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//*[@role='dialog' and @aria-label='Add customer']")))

    # Fill only fields that really exist (placeholders verified in CustomerFormModal.jsx)
    name_input = driver.find_element(By.XPATH, "//input[@placeholder='e.g. Rafiq Ahmed']")
    name_input.clear()
    name_input.send_keys("Selenium Test Customer")
    phone_input = driver.find_element(By.XPATH, "//input[@placeholder='+880 1711-234567']")
    phone_input.clear()
    phone_input.send_keys("01700000000")
    email_input = driver.find_element(By.XPATH, "//input[@placeholder='customer@example.com']")
    email_input.clear()
    email_input.send_keys("selenium.test.customer@example.com")

    # Save (primary button text "Add Customer", verified in CustomerFormModal.jsx)
    driver.find_element(By.XPATH, "//button[contains(@class, 'cust-btn-primary')]").click()
    time.sleep(3)

    # Dialog must close; verify the new customer appears via the real search field
    assert not driver.find_elements(By.XPATH, "//*[@role='dialog' and @aria-label='Add customer']"), \
        "Add dialog did not close — save may have failed."
    search = wait.until(EC.visibility_of_element_located((By.XPATH, "//input[@aria-label='Search customers by name, phone or email']")))
    search.clear()
    search.send_keys("Selenium Test Customer")
    time.sleep(2)
    body = driver.find_element(By.TAG_NAME, "body").text
    assert "Selenium Test Customer" in body and "No customers found" not in body

    print("Current URL:", driver.current_url)
    print("New customer found in list: Selenium Test Customer")
    print("PASS: Customer Add")
except Exception as e:
    print("FAIL: Customer Add")
    print("Error:", e)
    driver.save_screenshot("23_customer_add_FAIL.png")

In [ ]:
driver.quit()